# NYISO Solar Forecasting: Hyperparameter Tuning

## Imports and Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
random_state = 42

In [ ]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

model_ready_in = processed_dir / "04_system_model_ready_data.csv"

split_date = pd.Timestamp("2024-07-01 00:00:00+00:00")
validation_start = pd.Timestamp("2024-01-01 00:00:00+00:00")

In [ ]:
df_model = pd.read_csv(model_ready_in, low_memory=False)

df_model.columns = (
    df_model.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_model["time_stamp"] = pd.to_datetime(df_model["time_stamp"], utc=True, errors="coerce")
df_model["time_local"] = df_model["time_stamp"].dt.tz_convert("America/New_York")

In [ ]:
target = "forecast_error_mw"

required_cols = [
    "time_stamp",
    "time_local",
    "zone_name",
    "dataset_split",
    "actual_mw",
    "forecast_mw",
    "forecast_error_mw",
]

missing_required = [c for c in required_cols if c not in df_model.columns]
if missing_required:
    raise ValueError(f"Missing Necessary Columns in Dataset: {missing_required}")

df_model["hour_local"] = df_model["time_local"].dt.hour
df_model["month_local"] = df_model["time_local"].dt.month
df_model["dayofyear_local"] = df_model["time_local"].dt.dayofyear
df_model["is_daylight"] = (df_model["shortwave_radiation"] > 0).astype(int)

feature_cols = [c for c in df_model.columns if c not in required_cols + [
    "hour_local",
    "month_local",
    "dayofyear_local",
    "is_daylight",
]]

if "forecast_mw" not in feature_cols:
    feature_cols = ["forecast_mw"] + feature_cols

print("\nTarget:", target)
print("Number of Features:", len(feature_cols))
print(feature_cols)

In [ ]:
train_end = df_model.loc[df_model["dataset_split"].eq("train"), "time_stamp"].max()
test_start = df_model.loc[df_model["dataset_split"].eq("test"), "time_stamp"].min()

X = df_model[feature_cols].copy()
y = df_model[target].copy()

train_mask = (
    df_model["dataset_split"].eq("train")
    & y.notna()
)

test_mask = (
    df_model["dataset_split"].eq("test")
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

subtrain_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].lt(validation_start)
    & df_model[target].notna()
)

valid_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].ge(validation_start)
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

train_df = df_model.loc[train_mask].copy()
test_df = df_model.loc[test_mask].copy()
subtrain_df = df_model.loc[subtrain_mask].copy()
valid_df = df_model.loc[valid_mask].copy()

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
X_subtrain = subtrain_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()
y_subtrain = subtrain_df[target].copy()
y_valid = valid_df[target].copy()

baseline_actual_test = test_df["actual_mw"].copy()
baseline_forecast_test = test_df["forecast_mw"].copy()

baseline_actual_valid = valid_df["actual_mw"].copy()
baseline_forecast_valid = valid_df["forecast_mw"].copy()

daylight_test_mask = test_df["is_daylight"] == 1
daylight_valid_mask = valid_df["is_daylight"] == 1

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_subtrain.shape[0] == y_subtrain.shape[0]
assert X_valid.shape[0] == y_valid.shape[0]